# Phase 2 — LSTM with learned Q / R–style noise

Same probabilistic story as `phase2_transformer_learned_qr.ipynb`, using `ImprovedLSTMLearnedNoise` (decoder LSTM over history then autoregressive steps). The extra eight outputs per step are unconstrained logits mapped with **softplus** to positive variances for NLL training.

Use this notebook if you want a lighter recurrent baseline than the transformer, still with learnable uncertainty for MOT-style fusion.

In [ ]:
import os
import torch
from torch import optim
from torch.utils.data import DataLoader

from dataset import GTSequenceDataset
from learned_noise_motion import ImprovedLSTMLearnedNoise, LearnedNoiseMotionLoss

SEQ_IN_LEN = 30
SEQ_OUT_LEN = 20
SEQ_TOTAL_LEN = 50
BATCH_SIZE = 512
STEPS = 4
NOISE_COEFFICIENT = 0.15
NOISE_PROB = 0.2
BASE_DIR = os.environ.get("MOT_DATASET_ROOT", "../../Datasets/")

train_dataset = GTSequenceDataset.from_roots(
    [f"{BASE_DIR}MOT17/train"],
    seq_in_len=SEQ_IN_LEN,
    seq_out_len=SEQ_OUT_LEN,
    seq_total_len=SEQ_TOTAL_LEN,
    steps=STEPS,
    noise_coeff=NOISE_COEFFICIENT,
    noise_prob=NOISE_PROB,
)
val_dataset = GTSequenceDataset.from_roots(
    [f"{BASE_DIR}MOT17/val"],
    seq_in_len=SEQ_IN_LEN,
    seq_out_len=SEQ_OUT_LEN,
    seq_total_len=SEQ_TOTAL_LEN,
    steps=STEPS,
    noise_coeff=NOISE_COEFFICIENT,
    noise_prob=NOISE_PROB,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LR = 3e-4
NUM_EPOCHS = 20

model = ImprovedLSTMLearnedNoise(
    input_dim=13,
    d_model=256,
    hidden_dim=256,
    num_layers=2,
    dropout=0.1,
    teacher_forcing_ratio=0.5,
).to(DEVICE)

criterion = LearnedNoiseMotionLoss(
    nll_coeff=1.0,
    ciou_coeff=0.5,
    conf_coeff=0.25,
    r_supervise_coeff=0.1,
)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(NUM_EPOCHS, 1))

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,} | device={DEVICE}")

In [ ]:
best = float("inf")
os.makedirs("pretrained", exist_ok=True)
ckpt = "pretrained/lstm_learned_qr.pth"

for epoch in range(1, NUM_EPOCHS + 1):
    tr = model.train_one_epoch(train_loader, optimizer, criterion, device=DEVICE)
    va = model.evaluate(val_loader, criterion, device=DEVICE)
    scheduler.step()
    if va < best:
        best = va
        model.save_weight(ckpt)
    print(f"epoch {epoch:03d}  train {tr:.5f}  val {va:.5f}  lr {scheduler.get_last_lr()[0]:.2e}")

print("best val loss:", best, "saved:", ckpt)

In [ ]:
model.eval()
src, trg, _, gt_trg = next(iter(val_loader))
src, trg = src.to(DEVICE), trg.to(DEVICE)
steps = trg.size(1) - 1
with torch.no_grad():
    pred, _, _ = model.inference(src, trg[:, :1, :], num_steps=steps)
print("teacher-forced val loss uses train_one_epoch/evaluate; this cell is pure AR length", pred.shape)